# Rung 32 — reachability gate: does `--freeze_aligner false` reach the merger?

**The run generator.** Config inline; the engine is `_tools/reachability_smoke.py`, imported.

🔴 **This is NOT the rung.** It is the gate that decides whether the rung is one flag, run
before any GPU is spent on a full arm.

The G-COV census read the adapter as **720 tensors = 504 LLM + 216 ViT + 0 aligner**. Two
causes give that zero: nothing under the aligner prefixes is a `Linear`, or `freeze_aligner`
excludes them. Model inspection on 2026-08-08 settled the first half — the merger owns
**8 Linear layers** (`model.visual.merger.linear_fc{1,2}` plus
`deepstack_merger_list.{0,1,2}.linear_fc{1,2}`, i.e. **four merger blocks**). Whether the flag
lands LoRA on them only a real ms-swift run can say.

**PASS = `n_aligner > 0`** on the treatment leg **and `n_aligner == 0`** on the control leg.
Without the control a pass could be an artifact of the prefix classifier rather than the flag.

```bash
cd /workspace/repo_leo/experiments/32-aligner-unfreeze
papermill 32_reachability.ipynb /workspace/tmp/leo_32_unfrozen.ipynb -p FREEZE_ALIGNER False --log-output
papermill 32_reachability.ipynb /workspace/tmp/leo_32_frozen.ipynb   -p FREEZE_ALIGNER True  --log-output
```

In [ ]:
# papermill parameters
FREEZE_ALIGNER = False   # False = treatment (merger trains) · True = control leg
MAX_STEPS = 5
ROWS = 32

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "_tools"))

from reachability_smoke import Config, smoke

cfg = Config(freeze_aligner=bool(FREEZE_ALIGNER), max_steps=MAX_STEPS, rows=ROWS)
print(f"freeze_aligner={cfg.freeze_aligner} -> {cfg.run_dir}")
result = smoke(cfg)
print(json.dumps(result.get("gcov", {"no gcov": result.get('reason')}), indent=2)[:1200])